# 08 — IPS, SNIPS, clipping (T6)


In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))
from src.dataset import OpenBanditDataset
from obp.ope import (
    InverseProbabilityWeighting,
    SelfNormalizedInverseProbabilityWeighting,
    OffPolicyEvaluation,
)
from src.propensity import (
    train_propensity_model,
    extract_propensity_scores,
    effective_sample_size,
)
from src.estimators import ips_with_clipping, snips_estimate, clipping_experiment

sns.set_theme(style="whitegrid")
np.random.seed(42)

N_ACTIONS = 80
N_FEATURES = 30
LEN_LIST = 3
RANDOM_STATE = 42
BOOTSTRAP_SAMPLES = 200
PI_EVAL = 1.0 / N_ACTIONS  # uniform evaluation policy

FIGURES_DIR = Path("../figures/week6")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Wczytanie danych i pscores


In [ ]:
dataset_random = OpenBanditDataset(behavior_policy="random", campaign="all")
feedback_random = dataset_random.obtain_batch_bandit_feedback()

dataset_bts = OpenBanditDataset(behavior_policy="bts", campaign="all")
feedback_bts = dataset_bts.obtain_batch_bandit_feedback()

context_random = feedback_random["context"][:, :N_FEATURES]
action_random  = feedback_random["action"]

context_bts = feedback_bts["context"][:, :N_FEATURES]
action_bts  = feedback_bts["action"]
reward_bts  = feedback_bts["reward"].astype(np.float64)
position_bts = feedback_bts["position"]

PSCORES_PATH = Path("../results/week5/pscores_bts.npy")
if PSCORES_PATH.exists():
    pscores_bts = np.load(PSCORES_PATH)
    print(f"Loaded pscores from {PSCORES_PATH.resolve()}")
else:
    print("pscores_bts.npy not found — training propensity model (run 07 first to skip this)")
    pscore_model = train_propensity_model(context_random, action_random, N_ACTIONS, RANDOM_STATE)
    pscores_bts = extract_propensity_scores(pscore_model, context_bts, action_bts)

print(f"pscores_bts — shape: {pscores_bts.shape}")
print(f"  min={pscores_bts.min():.6f}  max={pscores_bts.max():.6f}  mean={pscores_bts.mean():.4f}")
print(f"  % NaN: {np.isnan(pscores_bts).mean()*100:.2f}%")


## IPS i SNIPS — punkt bazowy (bez clippingu)


In [ ]:
v_ips_base  = ips_with_clipping(reward_bts, pscores_bts, PI_EVAL, clip_lambda=None)
v_snips_base = snips_estimate(reward_bts, pscores_bts, PI_EVAL, clip_lambda=None)

print(f"V_IPS  (unclipped): {v_ips_base:.6f}")
print(f"V_SNIPS (unclipped): {v_snips_base:.6f}")

weights_base = PI_EVAL / np.clip(pscores_bts, 1e-9, None)
ess, ess_ratio = effective_sample_size(weights_base)
print(f"\nESS: {ess:.0f} / {len(weights_base)} = {ess_ratio:.4f}")


## Bootstrap CI przez bibliotekę OBP


In [ ]:
ips_estimator  = InverseProbabilityWeighting()
snips_estimator = SelfNormalizedInverseProbabilityWeighting()

n_rounds_bts = int(feedback_bts["n_rounds"])
action_dist_eval = np.full(
    (n_rounds_bts, N_ACTIONS, LEN_LIST),
    PI_EVAL,
    dtype=np.float64,
)

feedback_bts_ope = {**feedback_bts, "pscore": pscores_bts.astype(np.float64)}

ope = OffPolicyEvaluation(
    bandit_feedback=feedback_bts_ope,
    ope_estimators=[ips_estimator, snips_estimator],
)

ci_results = ope.estimate_intervals(
    action_dist=action_dist_eval,
    n_bootstrap_samples=BOOTSTRAP_SAMPLES,
    random_state=RANDOM_STATE,
)

for name, ci in ci_results.items():
    print(f"{name.upper():6s}  mean={ci['mean']:.6f}  "
          f"CI=[{ci['95.0% CI (lower)']:.6f}, {ci['95.0% CI (upper)']:.6f}]")


## Eksperymenty z clippingiem — bias-variance tradeoff


In [ ]:
CLIP_LAMBDAS = [None, 0.001, 0.01, 0.1, 1.0, 5.0, 10.0, 50.0]

clip_results = clipping_experiment(
    reward=reward_bts,
    pscore_log=pscores_bts,
    pi_eval=PI_EVAL,
    lambdas=CLIP_LAMBDAS,
    n_bootstrap=BOOTSTRAP_SAMPLES,
    random_state=RANDOM_STATE,
)

ess_per_lambda = {}
for lam in CLIP_LAMBDAS:
    denom = np.clip(pscores_bts, lam, None) if lam is not None else pscores_bts
    w = PI_EVAL / np.clip(denom, 1e-9, None)
    ess_val, ess_r = effective_sample_size(w)
    ess_per_lambda[lam] = {"ess": ess_val, "ess_ratio": ess_r}

rows = []
for lam in CLIP_LAMBDAS:
    r = clip_results[lam]
    e = ess_per_lambda[lam]
    rows.append({
        "lambda": str(lam),
        "V_IPS": r["ips_mean"],
        "IPS_std": r["ips_std"],
        "IPS_CI_low": r["ips_ci_lower"],
        "IPS_CI_high": r["ips_ci_upper"],
        "V_SNIPS": r["snips_mean"],
        "SNIPS_std": r["snips_std"],
        "ESS": e["ess"],
        "ESS_ratio": e["ess_ratio"],
    })

clip_df = pd.DataFrame(rows)
print(clip_df.to_string(index=False, float_format="{:.6f}".format))


In [ ]:
lambdas_numeric = [lam if lam is not None else 0.0001 for lam in CLIP_LAMBDAS]
lambda_labels   = [str(lam) if lam is not None else "None" for lam in CLIP_LAMBDAS]

v_ips_list   = [clip_results[lam]["ips_mean"] for lam in CLIP_LAMBDAS]
std_ips_list = [clip_results[lam]["ips_std"] for lam in CLIP_LAMBDAS]
ess_list     = [ess_per_lambda[lam]["ess_ratio"] for lam in CLIP_LAMBDAS]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

x = np.arange(len(CLIP_LAMBDAS))
ax1.plot(x, v_ips_list, "o-", color="steelblue", label="V_IPS (bias proxy)")
ax1.fill_between(
    x,
    np.array(v_ips_list) - np.array(std_ips_list),
    np.array(v_ips_list) + np.array(std_ips_list),
    alpha=0.2, color="steelblue",
)
ax2.bar(x, ess_list, alpha=0.3, color="orange", label="ESS ratio (variance proxy)")

ax1.set_xticks(x)
ax1.set_xticklabels(lambda_labels, rotation=30)
ax1.set_xlabel("Clipping threshold λ")
ax1.set_ylabel("V_IPS", color="steelblue")
ax2.set_ylabel("ESS ratio", color="orange")
ax1.set_title("Bias-Variance Tradeoff vs Clipping Threshold λ")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bias_variance_clipping.png", dpi=160)
plt.show()


## Tabela porównawcza: DM vs IPS vs SNIPS


In [ ]:
V_DM_BTS    = 0.082847
V_DM_CI_LOW = 0.080344
V_DM_CI_HI  = 0.085181

ci_ips   = ci_results["ipw"]
ci_snips = ci_results["snipw"]

comparison = pd.DataFrame([
    {
        "Estimator": "Direct Method (DM)",
        "V̂": V_DM_BTS,
        "CI_lower": V_DM_CI_LOW,
        "CI_upper": V_DM_CI_HI,
        "CI_width": V_DM_CI_HI - V_DM_CI_LOW,
        "Note": "Bias od reward modelu",
    },
    {
        "Estimator": "IPS",
        "V̂": ci_ips["mean"],
        "CI_lower": ci_ips["95.0% CI (lower)"],
        "CI_upper": ci_ips["95.0% CI (upper)"],
        "CI_width": ci_ips["95.0% CI (upper)"] - ci_ips["95.0% CI (lower)"],
        "Note": "Wysoka wariancja (brak clippingu)",
    },
    {
        "Estimator": "SNIPS",
        "V̂": ci_snips["mean"],
        "CI_lower": ci_snips["95.0% CI (lower)"],
        "CI_upper": ci_snips["95.0% CI (upper)"],
        "CI_width": ci_snips["95.0% CI (upper)"] - ci_snips["95.0% CI (lower)"],
        "Note": "Normalizowane wagi — mniejsza wariancja",
    },
])

print(comparison.to_string(index=False, float_format="{:.6f}".format))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

estimators = comparison["Estimator"]
means      = comparison["V̂"]
err_low    = means - comparison["CI_lower"]
err_high   = comparison["CI_upper"] - means

colors = ["steelblue", "darkorange", "seagreen"]
ax.barh(estimators, means, xerr=[err_low, err_high], color=colors, alpha=0.85, capsize=6)
ax.set_xlabel("Estimated policy value V̂")
ax.set_title("Week 6: DM vs IPS vs SNIPS — policy value comparison (95% CI)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dm_ips_snips_comparison.png", dpi=160)
plt.show()
